# 01. OpenVLA Setup, Weight Download, Inference Smoke Test

**런타임**: Colab → Runtime → Change runtime type → **A100** (권장) 또는 T4 (느림)

이 노트북이 끝나면:
1. Google Drive에 `openvla-7b` 가중치 저장 (~15GB) — 이후 노트북들이 재사용
2. 샘플 이미지 + 명령으로 action 7-DOF 출력 확인
3. **hidden state 추출 함수** 동작 확인 — 우리 연구의 가장 중요한 입력
4. **사람 영상 vs 로봇 영상의 hidden state cosine similarity** pilot 실험

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. Google Drive 마운트 (가중치 영구 저장용)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/openvla-pose'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/hidden_states', exist_ok=True)
print('Drive ready at', PROJECT_ROOT)

## 2. 의존성 설치

OpenVLA의 핵심: `transformers`, `timm`, `tokenizers`. flash-attn은 A100에서만.

In [ ]:
!pip install -q torch==2.2.0 torchvision==0.17.0
!pip install -q transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 accelerate==0.27.2
!pip install -q peft==0.11.1 bitsandbytes==0.43.1
!pip install -q sentencepiece einops draccus==0.8.0
# flash-attn은 A100/H100에서만 잘 컴파일됩니다. 실패해도 OpenVLA는 동작합니다.
!pip install -q flash-attn==2.5.5 --no-build-isolation || echo 'flash-attn 실패 — 무시'

## 3. OpenVLA 모델 로드

Hugging Face에서 자동 다운로드 + Drive에 캐시. 첫 실행은 10-15분 (네트워크).

**중요**: `cache_dir`을 Drive로 지정해서 재실행 시 다운로드 스킵.

In [ ]:
import torch
from transformers import AutoModelForVision2Seq, AutoProcessor

HF_CACHE = f'{PROJECT_ROOT}/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE

MODEL_ID = 'openvla/openvla-7b'

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, cache_dir=HF_CACHE)
vla = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    attn_implementation='flash_attention_2' if torch.cuda.get_device_capability()[0] >= 8 else 'eager',
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    cache_dir=HF_CACHE,
).to('cuda')

print('VLA loaded. Total params:', sum(p.numel() for p in vla.parameters()) / 1e9, 'B')

## 4. 추론 Smoke Test

OpenVLA는 BridgeData V2 스타일 입력을 기대합니다: 224x224 이미지 + `In: What action should the robot take to {INSTRUCTION}? Out:` 형식 프롬프트.

In [ ]:
from PIL import Image
import requests
from io import BytesIO

# 샘플 이미지 (BridgeData 스타일 — 위에서 본 부엌 씬)
url = 'https://github.com/openvla/openvla/raw/main/prismatic/conf/sample_image.jpg'
try:
    image = Image.open(BytesIO(requests.get(url, timeout=10).content)).convert('RGB')
except Exception:
    # fallback: 더미 이미지
    image = Image.new('RGB', (224, 224), color=(128, 128, 128))
    print('네트워크 이슈 → 더미 이미지 사용')

INSTRUCTION = 'pick up the red cup'
prompt = f'In: What action should the robot take to {INSTRUCTION}?\nOut:'

inputs = processor(prompt, image).to('cuda', dtype=torch.bfloat16)
action = vla.predict_action(**inputs, unnorm_key='bridge_orig', do_sample=False)
print('Predicted action (7-DOF: dx, dy, dz, drx, dry, drz, gripper):')
print(action)

## 5. Hidden State 추출 — 우리 연구의 핵심 입력

OpenVLA 내부에서 LLM의 마지막 hidden state를 꺼냅니다. 이게 우리가 'pose head'로 매핑할 표현입니다.

In [ ]:
@torch.inference_mode()
def extract_hidden_state(image, instruction, layer=-1):
    """image (PIL), instruction (str) → torch.Tensor [hidden_dim]
    layer: -1 = 마지막 layer, -2 = 끝에서 두 번째, ..."""
    prompt = f'In: What action should the robot take to {instruction}?\nOut:'
    inputs = processor(prompt, image).to('cuda', dtype=torch.bfloat16)
    outputs = vla(
        **inputs,
        output_hidden_states=True,
        return_dict=True,
    )
    # outputs.hidden_states: tuple of (B, T, D)
    h = outputs.hidden_states[layer]
    # 마지막 토큰의 hidden state를 representative로 사용
    return h[0, -1, :].float().cpu()

h = extract_hidden_state(image, INSTRUCTION)
print('Hidden state shape:', h.shape, 'dtype:', h.dtype)
print('Norm:', h.norm().item(), 'Mean:', h.mean().item())

## 6. **PILOT 실험: Functional Equivalence 가설 검증**

원안의 가장 위험한 가정은 *'사람 영상과 로봇 영상이 같은 태스크면 hidden state가 가깝다'* 입니다.

이걸 본격 학습 전에 작은 데이터로 검증합니다.

준비물:
- 같은 태스크(예: '컵을 집는다')의 사람 손 이미지 1장
- 같은 태스크의 로봇 그리퍼 이미지 1장
- 다른 태스크('서랍을 연다')의 이미지 1장

기대: `cos(h_human_cup, h_robot_cup) > cos(h_human_cup, h_robot_drawer)`

**위 부등식이 성립하지 않으면**, OpenVLA hidden state만으로는 anchor가 약하다는 뜻 → 명시적 contrastive alignment loss를 Phase 2에 추가해야 함.

In [ ]:
# TODO: 실제 페어 데이터를 Drive에 올린 뒤 경로 수정
PAIRS = [
    # (image_path, instruction, label)
    # ('drive/.../human_cup.jpg', 'pick up the cup', 'cup'),
    # ('drive/.../robot_cup.jpg', 'pick up the cup', 'cup'),
    # ('drive/.../human_drawer.jpg', 'open the drawer', 'drawer'),
    # ('drive/.../robot_drawer.jpg', 'open the drawer', 'drawer'),
]

import torch.nn.functional as F

if not PAIRS:
    print('PAIRS가 비어있습니다. Drive에 페어 이미지 4장 올린 뒤 위 리스트 채우세요.')
else:
    feats = {}
    for path, inst, label in PAIRS:
        img = Image.open(path).convert('RGB')
        h = extract_hidden_state(img, inst)
        feats[(path, label)] = h

    keys = list(feats.keys())
    print('\nPairwise cosine similarity:')
    for i in range(len(keys)):
        for j in range(i+1, len(keys)):
            ki, kj = keys[i], keys[j]
            cs = F.cosine_similarity(feats[ki].unsqueeze(0), feats[kj].unsqueeze(0)).item()
            same = '✓ same task' if ki[1] == kj[1] else '✗ diff task'
            print(f'  {os.path.basename(ki[0]):30s} vs {os.path.basename(kj[0]):30s}  cos={cs:+.3f}  [{same}]')

# 같은 태스크 평균 cos > 다른 태스크 평균 cos 이면 가설 일단 성립

## 7. 다음 단계로 넘기기

hidden state extractor를 다음 노트북에서 재사용할 수 있게 Drive에 헬퍼로 저장.

In [ ]:
helper_code = '''
import torch
from transformers import AutoModelForVision2Seq, AutoProcessor
import os

PROJECT_ROOT = "/content/drive/MyDrive/openvla-pose"
HF_CACHE = f"{PROJECT_ROOT}/hf_cache"
os.environ["HF_HOME"] = HF_CACHE

def load_openvla(device="cuda", dtype=torch.bfloat16):
    processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True, cache_dir=HF_CACHE)
    vla = AutoModelForVision2Seq.from_pretrained(
        "openvla/openvla-7b",
        attn_implementation="eager",
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        cache_dir=HF_CACHE,
    ).to(device)
    return processor, vla

@torch.inference_mode()
def extract_hidden_state(processor, vla, image, instruction, layer=-1):
    prompt = f"In: What action should the robot take to {instruction}?\\nOut:"
    inputs = processor(prompt, image).to(vla.device, dtype=vla.dtype)
    outputs = vla(**inputs, output_hidden_states=True, return_dict=True)
    return outputs.hidden_states[layer][0, -1, :].float().cpu()
'''

with open(f'{PROJECT_ROOT}/openvla_helpers.py', 'w') as f:
    f.write(helper_code)
print('Saved helper to', f'{PROJECT_ROOT}/openvla_helpers.py')